In [ ]:
import os
import time
import json
import ast
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from data_import import load_guardian_years, load_guardian_year

In [ ]:
API_KEY = "[]"
BASE_URL = "https://content.guardianapis.com/search"


In [3]:
DATA_DIR = Path("data/raw/guardian")

In [4]:
SLEEP_SECONDS = 1.1
PAGE_SIZE = 200

In [5]:
def guardian_get(params, sleep_seconds=SLEEP_SECONDS):
    params = params.copy()
    params["api-key"] = API_KEY

    resp = requests.get(BASE_URL, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()

    time.sleep(sleep_seconds)

    if data.get("response", {}).get("status") != "ok":
        raise RuntimeError(f"Unexpected API response: {data}")

    return data["response"]

In [6]:
def build_params(from_date, to_date, page=1, page_size=PAGE_SIZE):
    return {
        "section": "commentisfree",
        "type": "article",
        "from-date": from_date,
        "to-date": to_date,
        "use-date": "published",
        "order-by": "oldest",
        "page-size": page_size,
        "page": page,
        "show-fields": "headline,trailText,bodyText,byline",
        "show-tags": "keyword,contributor",
    }

In [7]:
def parse_result(result):
    fields = result.get("fields", {}) or {}
    tags = result.get("tags", []) or []

    keywords = [
        tag.get("webTitle")
        for tag in tags
        if tag.get("type") == "keyword"
    ]

    contributors = [
        tag.get("webTitle")
        for tag in tags
        if tag.get("type") == "contributor"
    ]

    return {
        "id": result.get("id"),
        "title": fields.get("headline") or result.get("webTitle"),
        "summary": fields.get("trailText"),
        "date": result.get("webPublicationDate"),
        "url": result.get("webUrl"),
        "author_raw": fields.get("byline"),
        "contributors": contributors,
        "n_contributors": len(contributors),
        "keywords": keywords,
        "body_text": fields.get("bodyText"),
        "section_id": result.get("sectionId"),
        "section_name": result.get("sectionName"),
    }

In [8]:
def fetch_year(year):
    from_date = f"{year}-01-01"
    to_date = f"{year}-12-31"

    # First call: get number of pages
    first_response = guardian_get(build_params(from_date, to_date, page=1, page_size=PAGE_SIZE))
    total = first_response.get("total", 0)
    pages = first_response.get("pages", 0)

    print(f"{year}: {total:,} results across {pages} pages")

    rows = [parse_result(r) for r in first_response.get("results", [])]

    for page in range(2, pages + 1):
        if page % 10 == 0 or page == pages:
            print(f"  fetching page {page}/{pages}")

        response = guardian_get(build_params(from_date, to_date, page=page, page_size=PAGE_SIZE))
        rows.extend(parse_result(r) for r in response.get("results", []))

    df = pd.DataFrame(rows).drop_duplicates(subset=["id"]).reset_index(drop=True)
    df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
    df = df.sort_values("date").reset_index(drop=True)

    return df


In [9]:
def save_guardian_year(df_year, year, folder=DATA_DIR):
    df_save = df_year.copy()

    for col in ["keywords", "contributors"]:
        if col in df_save.columns:
            df_save[col] = df_save[col].apply(
                lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else "[]"
            )

    out_path = Path(folder) / f"guardian_opinion_{year}.csv"
    df_save.to_csv(out_path, index=False)
    print(f"Saved {out_path}")

In [11]:
years = list(range(2005, 2010))

dfs = {}

for year in years:
    df_year = fetch_year(year)
    dfs[year] = df_year

    save_guardian_year(df_year, year, folder=DATA_DIR)

    print(f"{year} rows: {len(df_year):,}")
    print()

2005: 3 results across 1 pages
Saved data/raw/guardian/guardian_opinion_2005.csv
2005 rows: 3

2006: 7,964 results across 40 pages
  fetching page 10/40
  fetching page 20/40
  fetching page 30/40
  fetching page 40/40
Saved data/raw/guardian/guardian_opinion_2006.csv
2006 rows: 7,964

2007: 11,190 results across 56 pages
  fetching page 10/56
  fetching page 20/56
  fetching page 30/56
  fetching page 40/56
  fetching page 50/56
  fetching page 56/56
Saved data/raw/guardian/guardian_opinion_2007.csv
2007 rows: 11,190

2008: 12,674 results across 64 pages
  fetching page 10/64
  fetching page 20/64
  fetching page 30/64
  fetching page 40/64
  fetching page 50/64
  fetching page 60/64
  fetching page 64/64
Saved data/raw/guardian/guardian_opinion_2008.csv
2008 rows: 12,671

2009: 10,535 results across 53 pages
  fetching page 10/53
  fetching page 20/53
  fetching page 30/53
  fetching page 40/53
  fetching page 50/53
  fetching page 53/53
Saved data/raw/guardian/guardian_opinion_2009.